In [1]:
#uid variance equal
import json
from datasets import Dataset
import os

with open("/scratch/hojinkim/uid-reasoning/src/outputs/runs.baselines/aime.deepseek-r1-distill-qwen-1.5b.direct/train.8.24,10:7-5.json", 'r') as f:
    data = json.load(f)

uid_metrics = [
    "uid_variance_equal"
]

# Store filtered data for SFT training
sft_data_highest = []
sft_data_lowest = []

for problem in data:
    problem_id = problem.get('id', 'unknown')
    correct_answer = problem.get('answer', '')
    
    # Find all outputs for this problem
    outputs = []
    i = 0
    while f'Output_{i}' in problem:
        output_key = f'Output_{i}'
        metrics_key = f'Metrics_{i}'
        uid_metrics_key = f'uid_metrics_{i}'
        pred_answer_key = f'Pred_Answer_{i}'
        
        if uid_metrics_key in problem:
            # Check if answer is correct
            pred_answer = problem.get(pred_answer_key, '')
            is_correct = pred_answer == correct_answer
            
            outputs.append({
                'index': i,
                'uid_metrics': problem[uid_metrics_key],
                'accuracy': problem[metrics_key].get('acc', 0),
                'exact_match': problem[metrics_key].get('em', 0),
                'f1': problem[metrics_key].get('f1', 0),
                'pred_answer': pred_answer,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'output_text': problem[output_key]
            })
        i += 1
    
    if not outputs:
        continue
    
    # Check if any output is correct
    correct_outputs = [out for out in outputs if out['is_correct']]
    
    if correct_outputs:
        # Among correct outputs, find the one with highest UID score for each metric
        best_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with highest UID score among correct answers
                highest_output = max(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, -float('inf'))
                )
                best_outputs[metric] = highest_output
                
        # Create SFT training entry with just the output text
        for metric, output in best_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_highest.append(sft_entry)
        
        worst_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with lowest UID score among correct answers
                lowest_output = min(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, float('inf'))
                )
                worst_outputs[metric] = lowest_output
        
        # Create SFT training entry with just the output text
        for metric, output in worst_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_lowest.append(sft_entry)
        
print(f"Original problems: {len(data)}")
print(f"Highest UID output texts: {len(sft_data_highest)}")
print(f"Lowest UID output texts: {len(sft_data_lowest)}")

# Save filtered data
os.makedirs("sft_data", exist_ok=True)
with open("sft_data/sft_data_variance_highest.json", "w") as f:
    json.dump(sft_data_highest, f, indent=4)
    
with open("sft_data/sft_data_variance_lowest.json", "w") as f:
    json.dump(sft_data_lowest, f, indent=4)

sft_dataset_highest = Dataset.from_list(sft_data_highest)
sft_dataset_lowest = Dataset.from_list(sft_data_lowest)

sft_dataset_highest.push_to_hub("talzoomanzoo/highest_uid_variance_sft_ds")
sft_dataset_lowest.push_to_hub("talzoomanzoo/lowest_uid_variance_sft_ds")

/home/hojinkim/miniconda3/envs/entropy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original problems: 500
Highest UID output texts: 375
Lowest UID output texts: 375


Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.44s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/lowest_uid_variance_sft_ds/commit/c31e6d06d027891727979b401279bdc76802cc39', commit_message='Upload dataset', commit_description='', oid='c31e6d06d027891727979b401279bdc76802cc39', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/lowest_uid_variance_sft_ds', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/lowest_uid_variance_sft_ds'), pr_revision=None, pr_num=None)

In [2]:
#uid gini equal
import json
from datasets import Dataset
import os

with open("/scratch/hojinkim/uid-reasoning/src/outputs/runs.baselines/aime.deepseek-r1-distill-qwen-1.5b.direct/train.8.24,10:7-5.json", 'r') as f:
    data = json.load(f)

uid_metrics = [
    "uid_gini_equal"
]

# Store filtered data for SFT training
sft_data_highest = []
sft_data_lowest = []

for problem in data:
    problem_id = problem.get('id', 'unknown')
    correct_answer = problem.get('answer', '')
    
    # Find all outputs for this problem
    outputs = []
    i = 0
    while f'Output_{i}' in problem:
        output_key = f'Output_{i}'
        metrics_key = f'Metrics_{i}'
        uid_metrics_key = f'uid_metrics_{i}'
        pred_answer_key = f'Pred_Answer_{i}'
        
        if uid_metrics_key in problem and metrics_key in problem:
            # Check if answer is correct
            pred_answer = problem.get(pred_answer_key, '')
            is_correct = pred_answer == correct_answer
            
            outputs.append({
                'index': i,
                'uid_metrics': problem[uid_metrics_key],
                'accuracy': problem[metrics_key].get('acc', 0),
                'exact_match': problem[metrics_key].get('em', 0),
                'f1': problem[metrics_key].get('f1', 0),
                'pred_answer': pred_answer,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'output_text': problem[output_key]
            })
        i += 1
    
    if not outputs:
        continue
    
    # Check if any output is correct
    correct_outputs = [out for out in outputs if out['is_correct']]
    
    if correct_outputs:
        # Among correct outputs, find the one with highest UID score for each metric
        best_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with highest UID score among correct answers
                highest_output = max(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, -float('inf'))
                )
                best_outputs[metric] = highest_output
                
        # Create SFT training entry with just the output text
        for metric, output in best_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_highest.append(sft_entry)
        
        worst_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with lowest UID score among correct answers
                lowest_output = min(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, float('inf'))
                )
                worst_outputs[metric] = lowest_output
        
        # Create SFT training entry with just the output text
        for metric, output in worst_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_lowest.append(sft_entry)
        
print(f"Original problems: {len(data)}")
print(f"Highest UID output texts: {len(sft_data_highest)}")
print(f"Lowest UID output texts: {len(sft_data_lowest)}")

# Save filtered data
os.makedirs("sft_data", exist_ok=True)
with open("sft_data/sft_data_gini_highest.json", "w") as f:
    json.dump(sft_data_highest, f, indent=4)
    
with open("sft_data/sft_data_gini_lowest.json", "w") as f:
    json.dump(sft_data_lowest, f, indent=4)

sft_dataset_highest = Dataset.from_list(sft_data_highest)
sft_dataset_lowest = Dataset.from_list(sft_data_lowest)

sft_dataset_highest.push_to_hub("talzoomanzoo/highest_uid_gini_sft_ds")
sft_dataset_lowest.push_to_hub("talzoomanzoo/lowest_uid_gini_sft_ds")

Original problems: 500
Highest UID output texts: 375
Lowest UID output texts: 375


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.67s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/lowest_uid_gini_sft_ds/commit/3d7f42828025b3e07c273ffe36e9ef30540ab899', commit_message='Upload dataset', commit_description='', oid='3d7f42828025b3e07c273ffe36e9ef30540ab899', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/lowest_uid_gini_sft_ds', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/lowest_uid_gini_sft_ds'), pr_revision=None, pr_num=None)

In [3]:
#uid shannon equal
import json
from datasets import Dataset
import os

with open("/scratch/hojinkim/uid-reasoning/src/outputs/runs.baselines/aime.deepseek-r1-distill-qwen-1.5b.direct/train.8.24,10:7-5.json", 'r') as f:
    data = json.load(f)

uid_metrics = [
    "uid_shannon_equal"
]

# Store filtered data for SFT training
sft_data_highest = []
sft_data_lowest = []

for problem in data:
    problem_id = problem.get('id', 'unknown')
    correct_answer = problem.get('answer', '')
    
    # Find all outputs for this problem
    outputs = []
    i = 0
    while f'Output_{i}' in problem:
        output_key = f'Output_{i}'
        metrics_key = f'Metrics_{i}'
        uid_metrics_key = f'uid_metrics_{i}'
        pred_answer_key = f'Pred_Answer_{i}'
        
        if uid_metrics_key in problem and metrics_key in problem:
            # Check if answer is correct
            pred_answer = problem.get(pred_answer_key, '')
            is_correct = pred_answer == correct_answer
            
            outputs.append({
                'index': i,
                'uid_metrics': problem[uid_metrics_key],
                'accuracy': problem[metrics_key].get('acc', 0),
                'exact_match': problem[metrics_key].get('em', 0),
                'f1': problem[metrics_key].get('f1', 0),
                'pred_answer': pred_answer,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'output_text': problem[output_key]
            })
        i += 1
    
    if not outputs:
        continue
    
    # Check if any output is correct
    correct_outputs = [out for out in outputs if out['is_correct']]
    
    if correct_outputs:
        # Among correct outputs, find the one with highest UID score for each metric
        best_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with highest UID score among correct answers
                highest_output = max(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, -float('inf'))
                )
                best_outputs[metric] = highest_output
                
        # Create SFT training entry with just the output text
        for metric, output in best_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_highest.append(sft_entry)
        
        worst_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with lowest UID score among correct answers
                lowest_output = min(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, float('inf'))
                )
                worst_outputs[metric] = lowest_output
        
        # Create SFT training entry with just the output text
        for metric, output in worst_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_lowest.append(sft_entry)
        
print(f"Original problems: {len(data)}")
print(f"Highest UID output texts: {len(sft_data_highest)}")
print(f"Lowest UID output texts: {len(sft_data_lowest)}")

# Save filtered data
os.makedirs("sft_data", exist_ok=True)
with open("sft_data/sft_data_shannon_highest.json", "w") as f:
    json.dump(sft_data_highest, f, indent=4)
    
with open("sft_data/sft_data_shannon_lowest.json", "w") as f:
    json.dump(sft_data_lowest, f, indent=4)

sft_dataset_highest = Dataset.from_list(sft_data_highest)
sft_dataset_lowest = Dataset.from_list(sft_data_lowest)

sft_dataset_highest.push_to_hub("talzoomanzoo/highest_uid_shannon_sft_ds")
sft_dataset_lowest.push_to_hub("talzoomanzoo/lowest_uid_shannon_sft_ds")

Original problems: 500
Highest UID output texts: 375
Lowest UID output texts: 375


Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.84s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/lowest_uid_shannon_sft_ds/commit/3f7326a60a0eae1a0560ee2d7e42bb9f5da6eb55', commit_message='Upload dataset', commit_description='', oid='3f7326a60a0eae1a0560ee2d7e42bb9f5da6eb55', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/lowest_uid_shannon_sft_ds', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/lowest_uid_shannon_sft_ds'), pr_revision=None, pr_num=None)